In [ ]:
## Day selection / notebook starting point


In [ ]:
## Loading and setting up the trajectory data

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

DATA_DIR = "data2019"

# -----------------------------------------
# Load definitions once
# -----------------------------------------
sys.path.append(DATA_DIR)
import definitions_2019 as bd


# -----------------------------------------
# Load day once
# -----------------------------------------
DAY = 4
traj_file = os.path.join(DATA_DIR, f"beetrajectories_{DAY:03d}.hdf")
df = pd.read_hdf(traj_file)


print("Experiment starts on:",bd.startday)
current_date = bd.startday + pd.Timedelta(days=DAY)
print("Current Date:",current_date)
cohort_df = pd.read_csv("daydatamat.csv")
cohort_df=cohort_df[cohort_df["Day number"]==DAY]

uid_to_age = dict(zip(cohort_df["Bee unique ID"], cohort_df["Age"]))
df["age"] = df["uid"].map(uid_to_age)
uid_to_cohort = dict(zip(cohort_df["Bee unique ID"], cohort_df["Cohort ID"]))
df["cohort"] = df["uid"].map(uid_to_cohort)


In [ ]:
## Inspect the loaded dataframe

df

In [ ]:
## Check the available bee age groups

print(df["age"].unique())

# 1. Frame-gap statistics

This section analyzes missing-frame gaps in the trajectory data, including gap-length distributions, recovery fractions, and bee displacement across gaps.


In [ ]:
## Calculate missing-frame gaps for each bee

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure sorted
df2 = df.sort_values(["uid", "framenum"])

all_gaps = []

for uid, grp in df2.groupby("uid"):
    frames = grp["framenum"]

    # Missing frames between consecutive detections
    gaps = frames.diff().dropna().astype(int) - 1

    # Keep only actual missing intervals
    gaps = gaps[gaps > 0]

    all_gaps.extend(gaps.tolist())

all_gaps = np.array(all_gaps)

print(f"Total missing intervals: {len(all_gaps)}")
print(f"Mean gap length: {all_gaps.mean():.2f}")
print(f"Median gap length: {np.median(all_gaps):.0f}")
print(f"Maximum gap length: {all_gaps.max()}")

plt.figure(figsize=(8,5))
plt.hist(all_gaps, bins=np.arange(1,101)-0.5)
plt.yscale("log")
plt.xlabel("Consecutive missing frames")
plt.ylabel("Count (log scale)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
## Plot the distribution of missing-frame gap lengths

gap_counts = pd.Series(all_gaps).value_counts().sort_index()

plt.figure(figsize=(8,5))
plt.plot(gap_counts.index, gap_counts.values, marker=".")
plt.yscale("log")
plt.xlabel("Gap length")
plt.ylabel("Count")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
## Calculate gap statistics and cumulative recovery fractions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Build gap list
all_gaps = []

for _, grp in df.sort_values(["uid", "framenum"]).groupby("uid"):
    gaps = grp["framenum"].diff().dropna().astype(int) - 1
    all_gaps.extend(gaps[gaps > 0])

all_gaps = np.asarray(all_gaps)

# Gap statistics
gap_counts = pd.Series(all_gaps).value_counts().sort_index()

stats = pd.DataFrame({
    "Gap length": gap_counts.index,
    "Gap count": gap_counts.values
})

stats["Gap fraction"] = stats["Gap count"] / stats["Gap count"].sum()
stats["Gap cumulative"] = stats["Gap fraction"].cumsum()

# Weight by number of missing frames
stats["Missing frames"] = stats["Gap length"] * stats["Gap count"]
stats["Missing fraction"] = (
    stats["Missing frames"] / stats["Missing frames"].sum()
)
stats["Missing cumulative"] = stats["Missing fraction"].cumsum()

print(stats.head(30))

In [ ]:
## Plot cumulative gap recovery statistics

plt.figure(figsize=(8,5))

plt.plot(
    stats["Gap length"],
    stats["Gap cumulative"],
    lw=2,
    label="Fraction of gaps recovered"
)

plt.plot(
    stats["Gap length"],
    stats["Missing cumulative"],
    lw=2,
    label="Fraction of missing frames recovered"
)

plt.xlabel("Maximum gap length to interpolate")
plt.ylabel("Cumulative fraction")
plt.xlim(1,50)
plt.ylim(0,1.02)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
## Print gap-recovery statistics for selected maximum gap sizes

thresholds = [1,2,3,5,10,15,20,30,40,50]

print(f"{'Max gap':>8} {'Gaps recovered':>18} {'Missing frames recovered':>28}")

for t in thresholds:
    s = stats[stats["Gap length"] <= t]

    gap_frac = s["Gap fraction"].sum()
    miss_frac = s["Missing fraction"].sum()

    print(f"{t:8d} {100*gap_frac:17.2f}% {100*miss_frac:27.2f}%")

In [ ]:
## Load the comb image

import displayfunctions as bp
import pickle
import gzip

zfilln = 3
comb_contents_dir = 'comb-contents-images2019/'
comb = pickle.load(gzip.open(comb_contents_dir+'comb_'+str(DAY).zfill(zfilln)+'.pklz','rb'))

# 2. Bee speed, orientation, and angular-speed statistics

This section calculates and analyzes bee kinematics and orientation, including linear speed, mean orientation, and angular speed as functions of bee age.


In [ ]:
## Calculate per-bee speed from frame-to-frame displacement

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
FRAMES_PER_SECOND = 3
N_MAX             = 10   # max frame gap to interpolate across

# ── Apply camera offset ───────────────────────────────────────────────────────
df_analysis = df.copy()
df_analysis.loc[df_analysis["camera"] == 0, "x"] += bd.xpixels
df_analysis = df_analysis.sort_values(["uid", "framenum"]).reset_index(drop=True)

# ── Compute per-bee frame-to-frame displacement ───────────────────────────────
df_analysis["dx"]     = df_analysis.groupby("uid")["x"].diff()
df_analysis["dy"]     = df_analysis.groupby("uid")["y"].diff()
df_analysis["dframe"] = df_analysis.groupby("uid")["framenum"].diff()

# Keep only gaps <= N_MAX (includes consecutive frames where dframe==1)
df_speeds = df_analysis[
    (df_analysis["dframe"] >= 1) &
    (df_analysis["dframe"] <= N_MAX)
].copy()

# Speed = displacement / time_elapsed
# time_elapsed = dframe / FRAMES_PER_SECOND
df_speeds["displacement"] = np.sqrt(df_speeds["dx"]**2 + df_speeds["dy"]**2)
df_speeds["time_elapsed"] = df_speeds["dframe"] / FRAMES_PER_SECOND
df_speeds["speed"]        = df_speeds["displacement"] / df_speeds["time_elapsed"]

# Drop jitter outliers at 99.9th percentile
speed_999 = df_speeds["speed"].quantile(0.999)
print(f"99.9th percentile speed: {speed_999:.1f} px/s — using as outlier cutoff")
df_speeds = df_speeds[df_speeds["speed"] <= speed_999].copy()

print(f"Speed rows: {len(df_speeds)}")
print(df_speeds["speed"].describe())

In [ ]:
## Analyze mean bee speed by age group

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Per-bee mean speed ────────────────────────────────────────────────────────
per_bee = df_speeds.groupby("uid")["speed"].mean().reset_index()
per_bee.columns = ["uid", "mean_speed"]

# Merge with age instead of cohort
per_bee = per_bee.merge(df[["uid", "age"]].drop_duplicates(), on="uid")

age_list = sorted(per_bee["age"].unique())
n_ages   = len(age_list)

# Create a dynamic color mapping for ages
age_colors = {a: plt.cm.tab10(i % 10) for i, a in enumerate(age_list)}

# Clip at 99th percentile
speed_99 = per_bee["mean_speed"].quantile(0.99)
df_plot  = per_bee[per_bee["mean_speed"] <= speed_99].copy()

print(f"99th percentile mean speed: {speed_99:.1f} px/s")
print(f"Total bees: {len(df_plot)}")

bins = np.linspace(0, speed_99, 60)  # linear bins — mean speeds are less skewed

# ── Overall distribution ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(df_plot["mean_speed"], bins=bins, color="steelblue", edgecolor="none", alpha=0.85)
ax.axvline(df_plot["mean_speed"].median(), color="red",    linestyle="--", linewidth=1.5, label=f"Median: {df_plot['mean_speed'].median():.1f} px/s")
ax.axvline(df_plot["mean_speed"].mean(),   color="orange", linestyle="--", linewidth=1.5, label=f"Mean:   {df_plot['mean_speed'].mean():.1f} px/s")
ax.set_xlabel("Mean speed per bee (px/s)")
ax.set_ylabel("Count")
ax.set_title("Overall Distribution of Per-bee Mean Speed (all ages)")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("mean_speed_distribution_overall_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Per-age plots ─────────────────────────────────────────────────────────────
n_cols = 4
n_rows = int(np.ceil(n_ages / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
fig.suptitle("Per-bee Mean Speed Distribution by Age", fontsize=14)
axes = axes.flatten()

for i, a in enumerate(age_list):
    ax  = axes[i]
    sub = df_plot[df_plot["age"] == a]["mean_speed"]

    ax.hist(sub, bins=bins, color=age_colors[a], edgecolor="none", alpha=0.85)
    ax.axvline(sub.median(), color="red",    linestyle="--", linewidth=1.5, label=f"Median: {sub.median():.1f}")
    ax.axvline(sub.mean(),   color="orange", linestyle="--", linewidth=1.5, label=f"Mean:   {sub.mean():.1f}")
    ax.set_xlabel("Mean speed per bee (px/s)")
    ax.set_ylabel("Count (bees)")
    ax.set_title(f"Age {a}  ")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("mean_speed_distribution_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Per-age summary stats ─────────────────────────────────────────────────────
print("\nPer-age mean speed statistics (px/s):")
summary = df_plot.groupby("age")["mean_speed"].agg(
    n_bees="count",
    mean="mean",
    median="median",
    std="std",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    p99=lambda x: x.quantile(0.99)
).round(1)
print(summary.to_string())

# ── Median speed vs age ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(summary.index, summary["median"], marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Age (days)")
ax.set_ylabel("Median speed (px/s)")
ax.set_title("Median Per-bee Speed vs Age")
ax.set_xticks(summary.index)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("median_speed_vs_age.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
## Analyze mean bee orientation by age group

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 1. Per-bee Mean Orientation (Circular Mean) ───────────────────────────────
print("Calculating per-bee mean orientation using vector components...")

# Extract age instead of cohort
df_theta = df[["uid", "age", "theta"]].copy()

# Convert angles to vectors
df_theta["cos_t"] = np.cos(df_theta["theta"])
df_theta["sin_t"] = np.sin(df_theta["theta"])

# Group by uid and average the vector components
per_bee_components = df_theta.groupby("uid")[["cos_t", "sin_t"]].mean().reset_index()

# Convert averaged vectors back to an angle in radians [-pi, pi]
per_bee_components["mean_theta"] = np.arctan2(per_bee_components["sin_t"], per_bee_components["cos_t"])

# Merge with age data
per_bee = per_bee_components[["uid", "mean_theta"]].merge(
    df_theta[["uid", "age"]].drop_duplicates(), on="uid"
)

age_list = sorted(per_bee["age"].unique())
n_ages   = len(age_list)

# Create a dynamic color mapping for ages
age_colors = {a: plt.cm.tab10(i % 10) for i, a in enumerate(age_list)}

print(f"Total bees processed: {len(per_bee)}")

# ── 2. Overall distribution ───────────────────────────────────────────────────
# Angles are perfectly bounded, so we use exactly 60 bins between -Pi and Pi
bins = np.linspace(-np.pi, np.pi, 60)

fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(per_bee["mean_theta"], bins=bins, color="mediumpurple", edgecolor="none", alpha=0.85)

# Calculate global circular mean
global_cos = np.cos(per_bee["mean_theta"]).mean()
global_sin = np.sin(per_bee["mean_theta"]).mean()
global_mean_theta = np.arctan2(global_sin, global_cos)

ax.axvline(global_mean_theta, color="orange", linestyle="--", linewidth=2, label=f"Circ. Mean: {global_mean_theta:.2f} rad")

ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels(['$-\pi$', '$-\pi/2$', '0', '$\pi/2$', '$\pi$'])
ax.set_xlabel("Mean Orientation per bee (radians)")
ax.set_ylabel("Count")
ax.set_title("Overall Distribution of Per-bee Mean Orientation (all ages)")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("mean_orientation_distribution_overall_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 3. Per-age plots ──────────────────────────────────────────────────────────
n_cols = 2
n_rows = int(np.ceil(n_ages / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
fig.suptitle("Per-bee Mean Orientation Distribution by Age", fontsize=14)
axes = axes.flatten()

for i, a in enumerate(age_list):
    ax  = axes[i]
    sub = per_bee[per_bee["age"] == a]["mean_theta"]

    ax.hist(sub, bins=bins, color=age_colors[a], edgecolor="none", alpha=0.85)

    # Age circular mean
    a_mean_theta = np.arctan2(np.sin(sub).mean(), np.cos(sub).mean())

    ax.axvline(a_mean_theta, color="orange", linestyle="--", linewidth=2, label=f"Circ. Mean: {a_mean_theta:.2f}")

    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xticklabels(['$-\pi$', '$-\pi/2$', '0', '$\pi/2$', '$\pi$'])
    ax.set_xlabel("Mean Orientation per bee (radians)")
    ax.set_ylabel("Count (bees)")
    ax.set_title(f"Age {a}  (n={len(sub)} bees)")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("mean_orientation_distribution_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 4. Per-age summary stats ──────────────────────────────────────────────────
print("\nPer-age mean orientation statistics:")

def circ_mean(x):
    """Calculates the circular mean of an array of angles"""
    return np.arctan2(np.sin(x).mean(), np.cos(x).mean())

def circ_variance(x):
    """
    Calculates circular variance (0 to 1). 
    0 means all bees face exactly the same way.
    1 means bees are completely randomly dispersed.
    """
    R = np.sqrt(np.cos(x).mean()**2 + np.sin(x).mean()**2)
    return 1 - R

summary = per_bee.groupby("age")["mean_theta"].agg(
    n_bees="count",
    circ_mean=circ_mean,
    circ_var=circ_variance
).round(3)

print(summary.to_string())

In [ ]:
## Calculate and analyze angular speed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
FRAMES_PER_SECOND = 3
N_MAX             = 10   # max frame gap to interpolate across

# ── 1. Compute frame-to-frame angular displacement ────────────────────────────
df_analysis = df.copy()
df_analysis = df_analysis.sort_values(["uid", "framenum"]).reset_index(drop=True)

# Calculate raw differences
df_analysis["dtheta_raw"] = df_analysis.groupby("uid")["theta"].diff()
df_analysis["dframe"]     = df_analysis.groupby("uid")["framenum"].diff()

# Correct for the wrap-around boundary [-pi, pi] using shortest angular path
df_analysis["dtheta"] = np.arctan2(np.sin(df_analysis["dtheta_raw"]), np.cos(df_analysis["dtheta_raw"]))

# Keep only gaps <= N_MAX
df_speeds = df_analysis[
    (df_analysis["dframe"] >= 1) &
    (df_analysis["dframe"] <= N_MAX)
].copy()

# ── 2. Calculate Angular Speed ────────────────────────────────────────────────
# time_elapsed = dframe / FRAMES_PER_SECOND
df_speeds["time_elapsed"]     = df_speeds["dframe"] / FRAMES_PER_SECOND
df_speeds["angular_velocity"] = df_speeds["dtheta"] / df_speeds["time_elapsed"]

# We take the absolute value (magnitude of turning) so left/right turns don't cancel out
df_speeds["angular_speed"]    = np.abs(df_speeds["angular_velocity"])

# Drop jitter outliers at 99.9th percentile
ang_speed_999 = df_speeds["angular_speed"].quantile(0.999)
print(f"99.9th percentile angular speed: {ang_speed_999:.2f} rad/s — using as outlier cutoff")
df_speeds = df_speeds[df_speeds["angular_speed"] <= ang_speed_999].copy()

print(f"Angular speed rows: {len(df_speeds)}")

# ── 3. Per-bee mean angular speed ─────────────────────────────────────────────
per_bee = df_speeds.groupby("uid")["angular_speed"].mean().reset_index()
per_bee.columns = ["uid", "mean_ang_speed"]

# Merge with age
per_bee = per_bee.merge(df[["uid", "age"]].drop_duplicates(), on="uid")

age_list = sorted(per_bee["age"].unique())
n_ages   = len(age_list)

# Create a dynamic color mapping for ages
age_colors = {a: plt.cm.tab10(i % 10) for i, a in enumerate(age_list)}

# Clip at 99th percentile for plotting
ang_speed_99 = per_bee["mean_ang_speed"].quantile(0.99)
df_plot  = per_bee[per_bee["mean_ang_speed"] <= ang_speed_99].copy()

print(f"99th percentile mean angular speed: {ang_speed_99:.2f} rad/s")
print(f"Total bees: {len(df_plot)}")

bins = np.linspace(0, ang_speed_99, 60)

# ── 4. Overall distribution ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(df_plot["mean_ang_speed"], bins=bins, color="seagreen", edgecolor="none", alpha=0.85)
ax.axvline(df_plot["mean_ang_speed"].median(), color="red",    linestyle="--", linewidth=1.5, label=f"Median: {df_plot['mean_ang_speed'].median():.2f} rad/s")
ax.axvline(df_plot["mean_ang_speed"].mean(),   color="orange", linestyle="--", linewidth=1.5, label=f"Mean:   {df_plot['mean_ang_speed'].mean():.2f} rad/s")
ax.set_xlabel("Mean angular speed per bee (rad/s)")
ax.set_ylabel("Count")
ax.set_title("Overall Distribution of Per-bee Mean Angular Speed (all ages)")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("mean_ang_speed_distribution_overall_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 5. Per-age plots ──────────────────────────────────────────────────────────
n_cols = 4
n_rows = int(np.ceil(n_ages / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
fig.suptitle("Per-bee Mean Angular Speed Distribution by Age", fontsize=14)
axes = axes.flatten()

for i, a in enumerate(age_list):
    ax  = axes[i]
    sub = df_plot[df_plot["age"] == a]["mean_ang_speed"]

    ax.hist(sub, bins=bins, color=age_colors[a], edgecolor="none", alpha=0.85)
    ax.axvline(sub.median(), color="red",    linestyle="--", linewidth=1.5, label=f"Median: {sub.median():.2f}")
    ax.axvline(sub.mean(),   color="orange", linestyle="--", linewidth=1.5, label=f"Mean:   {sub.mean():.2f}")
    ax.set_xlabel("Mean angular speed per bee (rad/s)")
    ax.set_ylabel("Count (bees)")
    ax.set_title(f"Age {a} ")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("mean_ang_speed_distribution_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 6. Per-age summary stats ──────────────────────────────────────────────────
print("\nPer-age mean angular speed statistics (rad/s):")
summary = df_plot.groupby("age")["mean_ang_speed"].agg(
    n_bees="count",
    mean="mean",
    median="median",
    std="std",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    p99=lambda x: x.quantile(0.99)
).round(3)
print(summary.to_string())

# ── Median speed vs age ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(summary.index, summary["median"], marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Age (days)")
ax.set_ylabel("Median angular speed (rad/s)")
ax.set_title("Median Per-bee Angular Speed vs Age")
ax.set_xticks(summary.index)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("median_speed_vs_age.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Bee spatial distribution and heatmaps

This section analyzes where bees are located in the comb, including overall activity, unique-bee occupancy, frame-window-specific spatial distributions, and cohort/age-dependent spatial occupancy.


In [ ]:
## Convert bee positions to spatial grid cells

df2 = df.copy()

# global x coordinate (camera correction)
df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

l = 50   # grid size in pixels (tune this)

nx = int(np.ceil(2 * bd.xpixels / l))
ny = int(np.ceil(bd.ypixels / l))

print("Grid:", nx, "x", ny)

df2["ix"] = (df2["x"] // l).astype(int)
df2["iy"] = (df2["y"] // l).astype(int)

# clip (safety)
df2["ix"] = df2["ix"].clip(0, nx - 1)
df2["iy"] = df2["iy"].clip(0, ny - 1)

In [ ]:
## Inspect the gridded dataframe

df2

In [ ]:
## Calculate spatial activity and unique-bee occupancy

activity = np.zeros((nx, ny), dtype=np.int64)

np.add.at(activity, (df2["ix"].values, df2["iy"].values), 1)


unique_visits = df2[["ix", "iy", "uid"]].drop_duplicates()
unique_count = np.zeros((nx, ny), dtype=np.int64)

np.add.at(unique_count,
          (unique_visits["ix"].values, unique_visits["iy"].values),
          1)


In [ ]:
## Plot spatial activity and occupancy heatmaps

import matplotlib.pyplot as plt

# Physical extent (IMPORTANT)
extent = [0, 2*bd.xpixels, bd.ypixels, 0]

fig, axs = plt.subplots(1, 2, figsize=(12, 7))

# Activity heatmap
im1 = axs[0].imshow(
    activity.T,
    origin="upper",
    extent=extent,
    aspect="auto",
    cmap="hot",
    interpolation="nearest"
)
axs[0].set_title("Activity (total visits)")
axs[0].set_xlabel("x (pixels)")
axs[0].set_ylabel("y (pixels)")
plt.colorbar(im1, ax=axs[0])

# Unique bees heatmap
im2 = axs[1].imshow(
    unique_count.T,
    origin="upper",
    extent=extent,
    aspect="auto",
    cmap="viridis",
    interpolation="nearest"
)
axs[1].set_title("Unique bees visited")
axs[1].set_xlabel("x (pixels)")
axs[1].set_ylabel("y (pixels)")
plt.colorbar(im2, ax=axs[1])

plt.tight_layout()
plt.show()

### Frame-window dominant cohort/age spatial analysis

The following function is the generalized version of the dominant cohort/age analysis. It takes an arbitrary `frame_start` and `frame_end`, constructs the spatial grid for only that time window, and calculates three versions of the dominant cohort map:

1. **Raw:** dominant cohort based on total trajectory observations.
2. **Per-bee:** observations normalized by the number of bees in each cohort.
3. **Fraction of cohort time:** observations normalized by the total observations made by that cohort.

The resulting cohort maps can then be converted to age maps and plotted. Changing the frame range allows the spatial distribution of the bee population to be compared across different parts of the day.

In [ ]:
## Build dominant cohort grids for a selected frame range

def compute_dominant_cohort_grids(df, bd, frame_start=None, frame_end=None, l=50):
    """
    Build cohort counters and compute dominant cohort grids (3 normalizations)
    for a given frame range.

    Parameters
    ----------
    df           : full dataframe with columns [uid, x, y, camera, cohort, age, framenum]
    bd           : definitions module (needs bd.xpixels, bd.ypixels)
    frame_start  : first frame to include (inclusive). None = no lower bound.
    frame_end    : last frame to include (inclusive). None = no upper bound.
    l            : grid cell size in pixels (default 50)

    Returns
    -------
    dominant_raw      : (nx, ny) dominant cohort grid — raw counts
    dominant_per_bee  : (nx, ny) dominant cohort grid — per bee
    dominant_fraction : (nx, ny) dominant cohort grid — fraction of cohort time
    cohort_to_age     : dict mapping cohort -> age
    age_list          : sorted list of ages
    extent            : [xmin, xmax, ymin, ymax] for imshow
    """

    # ── Frame filtering ───────────────────────────────────────────────────────
    df2 = df.copy()
    if frame_start is not None:
        df2 = df2[df2["framenum"] >= frame_start]
    if frame_end is not None:
        df2 = df2[df2["framenum"] <= frame_end]

    if df2.empty:
        raise ValueError(f"No data in frame range [{frame_start}, {frame_end}]")

    print(f"Frame range: {df2['framenum'].min()} – {df2['framenum'].max()} "
          f"({df2['framenum'].nunique()} frames, {len(df2)} rows)")

    # ── Camera correction + grid ──────────────────────────────────────────────
    df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

    nx = int(np.ceil(2 * bd.xpixels / l))
    ny = int(np.ceil(bd.ypixels / l))
    print(f"Grid: {nx} x {ny}  (cell size = {l}px)")

    df2["ix"] = (df2["x"] // l).astype(int).clip(0, nx - 1)
    df2["iy"] = (df2["y"] // l).astype(int).clip(0, ny - 1)

    extent = [0, 2 * bd.xpixels, 0, bd.ypixels]

    # ── Cohort counter ────────────────────────────────────────────────────────
    cohort_list   = sorted(df2["cohort"].unique())
    cohort_to_idx = {c: i for i, c in enumerate(cohort_list)}
    n_cohorts     = len(cohort_list)

    c_idx = df2["cohort"].map(cohort_to_idx).values.astype(np.int32)
    ix    = df2["ix"].values
    iy    = df2["iy"].values

    cohort_counter = np.zeros((nx, ny, n_cohorts), dtype=np.int32)
    np.add.at(cohort_counter, (ix, iy, c_idx), 1)

    total_counts = cohort_counter.sum(axis=2)
    empty_mask   = total_counts == 0

    # ── Helper ────────────────────────────────────────────────────────────────
    def argmax_to_grid(matrix):
        idx  = np.argmax(matrix, axis=2)
        grid = np.full((nx, ny), np.nan)
        for c, i in cohort_to_idx.items():
            grid[idx == i] = c
        grid[empty_mask] = np.nan
        return grid

    # ── 1. Raw ────────────────────────────────────────────────────────────────
    dominant_raw = argmax_to_grid(cohort_counter)

    # ── 2. Per-bee ────────────────────────────────────────────────────────────
    bees_per_cohort   = df2.groupby("cohort")["uid"].nunique()
    cohort_bee_counts = np.array([bees_per_cohort[c] for c in cohort_list],
                                  dtype=np.float64)
    cohort_bee_counts = np.where(cohort_bee_counts == 0, 1, cohort_bee_counts)
    dominant_per_bee  = argmax_to_grid(
        cohort_counter / cohort_bee_counts[np.newaxis, np.newaxis, :]
    )

    # ── 3. Fraction of cohort time ────────────────────────────────────────────
    cohort_totals     = cohort_counter.sum(axis=(0, 1))
    cohort_totals     = np.where(cohort_totals == 0, 1, cohort_totals)
    dominant_fraction = argmax_to_grid(
        cohort_counter / cohort_totals[np.newaxis, np.newaxis, :]
    )

    # ── Metadata ──────────────────────────────────────────────────────────────
    cohort_to_age = (
        df2.drop_duplicates("cohort")
           .set_index("cohort")["age"]
           .to_dict()
    )
    age_list = sorted(df2["age"].unique())

    return dominant_raw, dominant_per_bee, dominant_fraction, cohort_to_age, age_list, extent

In [ ]:
## Run the dominant-cohort spatial analysis

raw, per_bee, fraction, c2a, ages, extent = compute_dominant_cohort_grids(
    df, bd, frame_start = 0, frame_end= 32398*2*4
)

In [ ]:
## Plot dominant age maps

from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import displayfunctions as bp
import numpy as np

def cohort_grid_to_age_grid(dominant_grid, cohort_to_age):
    """Map a dominant-cohort grid to a dominant-age grid."""
    age_grid = np.full_like(dominant_grid, np.nan)
    for cohort, age in cohort_to_age.items():
        age_grid[dominant_grid == cohort] = age
    return age_grid

# ── Shared setup ──────────────────────────────────────────────────────────────
cohort_to_age = (
    df2.drop_duplicates("cohort")
       .set_index("cohort")["age"]
       .to_dict()
)
age_list  = sorted(df2["age"].unique())
cmap      = ListedColormap([plt.cm.tab20(i) for i in range(len(age_list))])
bounds    = np.array(age_list + [age_list[-1] + 1]) - 0.5
norm      = BoundaryNorm(bounds, cmap.N)
extent    = [0, 2 * bd.xpixels, 0, bd.ypixels]

titles    = ["Raw (total frames)", "Per-bee (frames / n_bees)", "Fraction of cohort time"]
grids     = [raw,per_bee,fraction]

# ── 2 rows: top = 3 dominant-age maps, bottom = 3 comb images ────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 14))

for col, (dominant_grid, title) in enumerate(zip(grids, titles)):
    age_grid = cohort_grid_to_age_grid(dominant_grid, cohort_to_age)

    # ── Top row: age heatmap ──────────────────────────────────────────────────
    ax = axes[0, col]
    im = ax.imshow(age_grid.T, origin="upper", extent=extent,
                   cmap=cmap, norm=norm,
                   interpolation="nearest", aspect="equal")
    ax.set_title(title)
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    cbar = fig.colorbar(im, ax=ax, ticks=age_list, shrink=0.8)
    cbar.set_label("Age (days)")

    # ── Bottom row: comb reference ────────────────────────────────────────────
    ax_comb = axes[1, col]
    bp.showcomb(comb, ax=ax_comb)
    ax_comb.set_title(f"Comb — {title}")
    ax_comb.set_aspect("equal")

# ── Shared legend across all columns ─────────────────────────────────────────
handles = [mpatches.Patch(color=plt.cm.tab10(i), label=f"Age {a}")
           for i, a in enumerate(age_list)]
fig.legend(handles=handles, loc="lower center", ncol=len(age_list),
           bbox_to_anchor=(0.5, 0.01), fontsize=9)

plt.suptitle("Dominant Age per Cell — Three Normalizations", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
## Calculate spatial occupancy for a selected frame range

df2 = df.copy()
frame_start= 32398*2*0
frame_end=32398*2*1
df2 = df2[(df2["framenum"] >= frame_start) & (df2["framenum"] <= frame_end)]

# global x coordinate (camera correction)
df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

l = 50   # grid size in pixels (tune this)

nx = int(np.ceil(2 * bd.xpixels / l))
ny = int(np.ceil(bd.ypixels / l))

print("Grid:", nx, "x", ny)

df2["ix"] = (df2["x"] // l).astype(int)
df2["iy"] = (df2["y"] // l).astype(int)

# clip (safety)
df2["ix"] = df2["ix"].clip(0, nx - 1)
df2["iy"] = df2["iy"].clip(0, ny - 1)

In [ ]:
## Calculate time spent per spatial cell for each age group

# ── Time spent distribution over cells, per age group ────────────────────────

# Frames per (age, cell) pair
age_residency = df2.groupby(["age", "ix", "iy"]).size().reset_index(name="frame_count")
total_per_age = age_residency.groupby("age")["frame_count"].sum().reset_index(name="total_frames")
age_residency = age_residency.merge(total_per_age, on="age")
age_residency["fraction"] = age_residency["frame_count"] / age_residency["total_frames"]

age_list = sorted(df2["age"].unique())
extent    = [0, 2 * bd.xpixels, 0, bd.ypixels]
# One subplot per age — heatmap of fraction of time spent in each cell
n_cols =7 
n_rows = int(np.ceil(len(age_list) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(40, 5 * n_rows))
axes = axes.flatten()

for i, age in enumerate(age_list):
    ax      = axes[i]
    sub     = age_residency[age_residency["age"] == age]
    
    grid    = np.full((nx, ny), np.nan)
    grid[sub["ix"].values, sub["iy"].values] = sub["fraction"].values

    im = ax.imshow(grid.T, origin="upper", extent=extent,
                   cmap="hot", interpolation="nearest", aspect="equal")
    ax.set_title(f"Age {age} ")
    ax.set_xlabel("x (pixels)"); ax.set_ylabel("y (pixels)")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

#plt.suptitle("Time Spent Distribution by Age Group", fontsize=14)
plt.tight_layout()
plt.show()